
# Lab 1 — ROS Melodic Foundations with TurtleSim

**ROS Version:** Melodic (Ubuntu 18.04)  

---

## Overview
In this lab you will install and validate **ROS Melodic on Ubuntu 18.04**, then use **turtlesim** as a controlled sandbox to practice core ROS communication patterns: **nodes, topics, services, and name remapping**.

### Learning outcomes
By the end of this lab, you will be able to:
- Explain (and verify) ROS concepts: **master, nodes, topics, services**
- Use ROS command-line tools to inspect a running system
- Use **rqt** to call services (spawn turtles, change pen settings)
- Control multiple turtles via **topic remapping**
- Create a **catkin** workspace and build a ROS package
- Run a **publisher node** in both **Python** and **C++** to move a turtle


---
## Part A — Install & Configure ROS Melodic (Ubuntu 18.04)

### A1) Install ROS Melodic Desktop-Full
Follow the official ROS Melodic installation steps for Ubuntu 18.04.

After installing, ensure your terminal auto-sources ROS:


In [ ]:

# Run in Ubuntu Terminal
echo "source /opt/ros/melodic/setup.bash" >> ~/.bashrc
source ~/.bashrc



### A2) Initialize rosdep


In [ ]:

# Run in Ubuntu Terminal
sudo apt update
sudo apt install python-rosdep
sudo rosdep init
rosdep update
rosdep update --rosdistro=melodic



### A3) Quick verification


In [ ]:

# Run in Ubuntu Terminal
printenv | grep ROS
which roscore
rosversion -d



---
## Part B — ROS System Bring-Up + TurtleSim

### B1) Start the ROS master


In [ ]:

# Terminal 1 (keep running)
roscore



### B2) Start TurtleSim


In [ ]:

# Terminal 2
rosrun turtlesim turtlesim_node



### B3) Drive turtle1 with keyboard teleop


In [ ]:

# Terminal 3
rosrun turtlesim turtle_teleop_key



### B4) Inspect the running graph (engineering checks)


In [ ]:

# Terminal 4
rosnode list
rostopic list
rostopic info /turtle1/cmd_vel
rostopic echo -n 1 /turtle1/pose



---
## Part C — Services and rqt (Spawn + Set Pen)

### C1) Open rqt and call `/spawn`
1. Run `rqt`
2. In the UI: **Plugins → Services → Service Caller**
3. Select `/spawn`
4. Spawn a turtle (example): x=2.0, y=2.0, theta=0.0, name=`turtle2`



In [ ]:

# Terminal
rqt



### C2) Change turtle1 pen using `/turtle1/set_pen`
In rqt Service Caller:
- Select `/turtle1/set_pen`
- Example values:
  - r=255, g=0, b=0, width=5, off=0
- Click **Call**



---
## Part D — Control Multiple Turtles (Remapping)

Run a second teleop remapped to turtle2:



In [ ]:

# Terminal 5
rosrun turtlesim turtle_teleop_key turtle1/cmd_vel:=turtle2/cmd_vel __name:=teleop_turtle2



Now:
- Terminal 3 controls `turtle1`
- Terminal 5 controls `turtle2` (the terminal must be focused to read the keyboard)



---
## Part E — Build Your Own ROS Package (catkin workspace)

### E1) Create a workspace


In [ ]:

# Run in Ubuntu Terminal
mkdir -p ~/catkin_ws/src
cd ~/catkin_ws
catkin_make
echo "source ~/catkin_ws/devel/setup.bash" >> ~/.bashrc
source ~/.bashrc



### E2) Create a package


In [ ]:

# Run in Ubuntu Terminal
cd ~/catkin_ws/src
catkin_create_pkg lab1_turtle roscpp rospy geometry_msgs turtlesim



---
## Part F — Programmatic Turtle Motion (Python | C++)

You will implement a simple open-loop motion primitive: publish velocity commands to draw a **square**.



### F1) Python node — `square_driver.py`

Create the file and make it executable:


In [ ]:

# Run in Ubuntu Terminal
mkdir -p ~/catkin_ws/src/lab1_turtle/scripts
nano ~/catkin_ws/src/lab1_turtle/scripts/square_driver.py
chmod +x ~/catkin_ws/src/lab1_turtle/scripts/square_driver.py



Paste the following Python code into `square_driver.py`:


In [ ]:

#!/usr/bin/env python
import rospy
from geometry_msgs.msg import Twist

def main():
    rospy.init_node("square_driver_py")

    # Publish velocity commands to turtle1 by default
    pub = rospy.Publisher("/turtle1/cmd_vel", Twist, queue_size=10)

    rate = rospy.Rate(10)  # 10 Hz
    msg = Twist()

    # Engineering-style constants (tune if needed)
    linear_speed = 1.0     # units/s (turtlesim units)
    angular_speed = 1.57   # rad/s ~ 90 deg/s
    forward_time = 2.0     # seconds per edge
    turn_time = 1.0        # seconds per corner

    rospy.sleep(1.0)  # let connections settle

    for _ in range(4):
        # Move forward
        msg.linear.x = linear_speed
        msg.angular.z = 0.0
        t_end = rospy.Time.now() + rospy.Duration(forward_time)
        while rospy.Time.now() < t_end and not rospy.is_shutdown():
            pub.publish(msg)
            rate.sleep()

        # Turn 90 degrees
        msg.linear.x = 0.0
        msg.angular.z = angular_speed
        t_end = rospy.Time.now() + rospy.Duration(turn_time)
        while rospy.Time.now() < t_end and not rospy.is_shutdown():
            pub.publish(msg)
            rate.sleep()

    # Stop
    msg.linear.x = 0.0
    msg.angular.z = 0.0
    pub.publish(msg)

if __name__ == "__main__":
    main()



Build and run:


In [ ]:

# Run in Ubuntu Terminal
cd ~/catkin_ws
catkin_make
rosrun lab1_turtle square_driver.py



### F2) C++ node — `square_driver.cpp`

Create the file:


In [ ]:

# Run in Ubuntu Terminal
mkdir -p ~/catkin_ws/src/lab1_turtle/src
nano ~/catkin_ws/src/lab1_turtle/src/square_driver.cpp



Paste the following C++ code into `square_driver.cpp`:


In [ ]:

#include <ros/ros.h>
#include <geometry_msgs/Twist.h>

int main(int argc, char** argv) {
  ros::init(argc, argv, "square_driver_cpp");
  ros::NodeHandle nh;

  ros::Publisher pub = nh.advertise<geometry_msgs::Twist>("/turtle1/cmd_vel", 10);
  ros::Rate rate(10);

  geometry_msgs::Twist msg;

  const double linear_speed = 1.0;   // units/s
  const double angular_speed = 1.57; // rad/s
  const double forward_time = 2.0;   // seconds
  const double turn_time = 1.0;      // seconds

  ros::Duration(1.0).sleep(); // allow publisher connections to establish

  for (int i = 0; i < 4 && ros::ok(); i++) {
    // Forward
    msg.linear.x = linear_speed;
    msg.angular.z = 0.0;
    ros::Time t_end = ros::Time::now() + ros::Duration(forward_time);
    while (ros::Time::now() < t_end && ros::ok()) {
      pub.publish(msg);
      rate.sleep();
    }

    // Turn
    msg.linear.x = 0.0;
    msg.angular.z = angular_speed;
    t_end = ros::Time::now() + ros::Duration(turn_time);
    while (ros::Time::now() < t_end && ros::ok()) {
      pub.publish(msg);
      rate.sleep();
    }
  }

  // Stop
  msg.linear.x = 0.0;
  msg.angular.z = 0.0;
  pub.publish(msg);

  return 0;
}



Update `CMakeLists.txt` (inside `~/catkin_ws/src/lab1_turtle/`) by adding:


In [ ]:

# Add these lines near the bottom of CMakeLists.txt
add_executable(square_driver_cpp src/square_driver.cpp)
target_link_libraries(square_driver_cpp ${catkin_LIBRARIES})
add_dependencies(square_driver_cpp ${catkin_EXPORTED_TARGETS})



Build and run:


In [ ]:

# Run in Ubuntu Terminal
cd ~/catkin_ws
catkin_make
rosrun lab1_turtle square_driver_cpp



---
## Part G — Engineering Task

You will create and control a **third turtle** with a **distinct pen color** and demonstrate you can route commands to it.

### Task requirements
1. Spawn `turtle3`.
2. Set `turtle3` pen color to **green** (g=255) with visible width.
3. Control `turtle3` using:
   - teleop with remapping, AND
   - one of your programmatic nodes (Python or C++) redirected to turtle3.

### CLI equivalents (optional)


In [ ]:

# Spawn turtle3 (example)
rosservice call /spawn 3.0 3.0 0.0 "turtle3"

# Set turtle3 pen to green
rosservice call /turtle3/set_pen 0 255 0 5 0


In [ ]:

# Remap teleop to turtle3
rosrun turtlesim turtle_teleop_key turtle1/cmd_vel:=turtle3/cmd_vel __name:=teleop_turtle3


In [ ]:

# Remap your Python node output to turtle3
rosrun lab1_turtle square_driver.py /turtle1/cmd_vel:=/turtle3/cmd_vel

# Remap your C++ node output to turtle3
rosrun lab1_turtle square_driver_cpp /turtle1/cmd_vel:=/turtle3/cmd_vel



---
## Troubleshooting quick hits

- **`rosrun` / `roscore` not found** → source ROS:
  ```bash
  source /opt/ros/melodic/setup.bash
  ```

- **Package not found** after building → source workspace:
  ```bash
  source ~/catkin_ws/devel/setup.bash
  ```

- Teleop controls wrong turtle → confirm remapping string and terminal focus (teleop reads keyboard from the active terminal).
